# DataMate — LoRA leave-one-industry-out (chỉ train)

Train 3 adapter, mỗi adapter **bỏ hẳn một ngành** khỏi nhãn (`lora/leave_out.py`, `evals/industries.yaml`):
`pharmacy` (nhà thuốc), `building` (vật liệu xây dựng), `restaurant` (quán ăn). Sau đó đo mỗi adapter trên đúng
ngành nó chưa thấy (`evals.run_eval --subset ind_<ngành>`) — phép đo gần nhất với loại shop mới ở public.

Cùng công thức với adapter đã nộp (rank 16, 2 epoch, lr 1e-4). Cần GPU ≥ 40 GB. Mỗi adapter ~1 giờ.

1. Upload vào `MyDrive/datamate/`: `train_qlora.py`, `labels_lo_pharmacy.jsonl`, `labels_lo_building.jsonl`,
   `labels_lo_restaurant.jsonl` (từ `lora/data/` trên máy local).
2. Chạy hết các cell. Colab ngắt thì chạy lại cell train: adapter đã xong được bỏ qua, adapter đang dở train tiếp
   từ checkpoint.
3. Adapter ở `MyDrive/datamate/lora-lo-<ngành>/`.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate "huggingface_hub[hf_transfer]"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
WORK = "/content/drive/MyDrive/datamate"
FOLDS = ["pharmacy", "building", "restaurant"]
!ls -la {WORK}
for f in FOLDS:
    !wc -l {WORK}/labels_lo_{f}.jsonl


In [ ]:
# Train lần lượt từng ngành (~1 giờ mỗi ngành). Chạy lại cell này sau khi Colab ngắt.
for f in FOLDS:
    out = f"{WORK}/lora-lo-{f}"
    if os.path.isfile(f"{out}/adapter_model.safetensors"):
        print("skip (done):", out)
        continue
    print("=== train without", f)
    !python {WORK}/train_qlora.py --data {WORK}/labels_lo_{f}.jsonl --base Qwen/Qwen3-14B --out {out} \
        --rank 16 --alpha 32 --epochs 2 --lr 1e-4 --max-len 2560 --grad-accum 16


In [ ]:
for f in FOLDS:
    !ls -la {WORK}/lora-lo-{f}
